In [1]:
import pandas as pd
from datetime import datetime
import os

# ==========================================
# 1. Configuration and Paths
# ==========================================

# Base Paths (Uncomment the active path)
BASE_PATH = '..' # Currently active path

# Files
INPUT_FILE = 'Tabela_consumo_Itapua_120m.csv'
OUTPUT_FILE = 'Tabela_consumo_Itapua_120m_por_mes.csv'

def main():
    print("--- Starting Monthly Consumption Aggregation (120 Months) ---")

    # ==========================================
    # 2. Load Data
    # ==========================================
    input_path = os.path.join(BASE_PATH, 'includes', 'dados', INPUT_FILE)
    print(f"Reading file: {input_path}")
    
    # Reading CSV with semicolon delimiter
    df = pd.read_csv(input_path, sep=';', quotechar='"')

    # ==========================================
    # 3. Data Processing
    # ==========================================
    
    # Convert AM_REFERENCIA to string to ensure proper formatting
    df['AM_REFERENCIA'] = df['AM_REFERENCIA'].astype(str)
    
    # Extract year and month from AM_REFERENCIA (format: YYYYMM)
    df['ano'] = df['AM_REFERENCIA'].str[:4].astype(int)
    df['mes'] = df['AM_REFERENCIA'].str[4:6].astype(int)
    
    # Create date column from AM_REFERENCIA (first day of the reference month)
    df['data'] = pd.to_datetime(df['ano'].astype(str) + '-' + df['mes'].astype(str) + '-01')

    # Aggregate by Reference Month/Year (AM_REFERENCIA)
    # Summing volume (HCLQTCON) and taking the first date from the reference
    result_df = df.groupby('AM_REFERENCIA').agg({
        'HCLQTCON': 'sum',
        'data': 'first' 
    }).reset_index()

    # Rename columns to target format
    result_df = result_df.rename(columns={
        'AM_REFERENCIA': 'mes/ano',
        'HCLQTCON': 'consumo'
    })

    # Sort by chronological order (using mes/ano numeric value)
    result_df = result_df.sort_values('mes/ano')

    # Reorder columns
    result_df = result_df[['mes/ano', 'consumo', 'data']]

    # ==========================================
    # 4. Save Results
    # ==========================================
    output_path = os.path.join(BASE_PATH, 'includes', OUTPUT_FILE)
    
    # Saving with semicolon separator
    result_df.to_csv(output_path, index=False, sep=';', quotechar='"', decimal=',')

    print(f"File successfully generated: {output_path}")
    print("\nSample Data (first 12 months):")
    print(result_df.head(-12))
    
if __name__ == "__main__":
    main()

--- Starting Monthly Consumption Aggregation (120 Months) ---
Reading file: ..\includes\dados\Tabela_consumo_Itapua_120m.csv


File successfully generated: ..\includes\Tabela_consumo_Itapua_120m_por_mes.csv

Sample Data (first 12 months):
    mes/ano  consumo       data
0    201501   202383 2015-01-01
1    201502   197577 2015-02-01
2    201503   202766 2015-03-01
3    201504   185592 2015-04-01
4    201505   189076 2015-05-01
..      ...      ...        ...
115  202408   169213 2024-08-01
116  202409   180076 2024-09-01
117  202410   173150 2024-10-01
118  202411   178233 2024-11-01
119  202412   181749 2024-12-01

[120 rows x 3 columns]
